In [2]:
!pip install transformers==4.35.2 timm==0.9.2

  Using cached transformers-4.35.2-py3-none-any.whl.metadata (123 kB)
  Using cached timm-0.9.2-py3-none-any.whl.metadata (68 kB)
  Using cached tokenizers-0.15.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusolver_cu12-11.6.1.9-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.3.1.170-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
Using cached transformers-4.35.2-py3-none-any.whl (7.9 MB)
Using cached timm-0.9.2-py3-none-any.whl (2.2 MB)
Using cached tokenizers-0.15.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.6 MB)
Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl (664.8 MB)
Using cached nvidia_cusolver_cu12-11.6.1.9-py3-none-manylinux2014_x86_64.whl (127.9 MB)
Using cached nvidia_cusparse_cu12-12.3.1.170-py3-none-manylinux2014_x86_64.whl (207.5 MB)

In [3]:
import os
import json
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from transformers import (
    CLIPVisionModel,
    BertTokenizer,
    BertModel
)

from torchvision import transforms
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE =", device)

/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


DEVICE = cuda


In [4]:
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.48145466, 0.4578275, 0.40821073],
        std=[0.26862954, 0.26130258, 0.27577711]
    ),
])

# =============================================================
# 2. DATASET CLASS
# =============================================================
class ENTMedClipDataset(Dataset):
    def __init__(self, img_dir, metadata_path):
        self.img_dir = img_dir
        self.items = []

        with open(metadata_path, "r") as f:
            for line in f:
                self.items.append(json.loads(line))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]

        img_path = os.path.join(self.img_dir, "images", item["image"])
        image = Image.open(img_path).convert("RGB")
        image = image_transform(image)

        text = item["text"]

        return {"image": image, "text": text}

In [5]:
def collate_fn(batch):
    images = torch.stack([x["image"] for x in batch])
    texts = [x["text"] for x in batch]
    return {"images": images, "texts": texts}

# =============================================================
# 4. ALIGN MODEL IMPLEMENTATION
# =============================================================
class ALIGNForRetrieval(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()

        # -------- Image Encoder (CLIP-ViT) --------
        self.image_encoder = CLIPVisionModel.from_pretrained(
            "openai/clip-vit-base-patch32"
        )
        vision_dim = self.image_encoder.config.hidden_size

        # -------- Text Encoder (BERT) --------
        self.text_encoder = BertModel.from_pretrained("bert-base-uncased")
        text_dim = self.text_encoder.config.hidden_size

        # -------- Projection Heads --------
        self.img_proj = nn.Linear(vision_dim, embed_dim)
        self.txt_proj = nn.Linear(text_dim, embed_dim)

        # -------- Tokenizer --------
        self.tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

    def encode_image(self, pixel_values):
        outputs = self.image_encoder(pixel_values=pixel_values)
        img = outputs.pooler_output
        img = self.img_proj(img)
        img = img / img.norm(dim=-1, keepdim=True)
        return img

    def encode_text(self, texts):
        tokens = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(self.txt_proj.weight.device)

        outputs = self.text_encoder(
            input_ids=tokens["input_ids"],
            attention_mask=tokens["attention_mask"]
        )

        txt = outputs.pooler_output
        txt = self.txt_proj(txt)
        txt = txt / txt.norm(dim=-1, keepdim=True)
        return txt

    def forward(self, images, texts):
        img_emb = self.encode_image(images)
        txt_emb = self.encode_text(texts)

        logits = img_emb @ txt_emb.T
        return logits, img_emb, txt_emb


In [12]:
root = "/kaggle/input/sumotosima/sumotoshima_processed"

train_ds = ENTMedClipDataset(f"{root}/train", f"{root}/train/metadata_train.jsonl")
val_ds   = ENTMedClipDataset(f"{root}/val",   f"{root}/val/metadata_val.jsonl")
test_ds  = ENTMedClipDataset(f"{root}/test",  f"{root}/test/metadata_test.jsonl")

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds,  batch_size=1,  shuffle=False, collate_fn=collate_fn)

print("Train =", len(train_ds))
print("Val =", len(val_ds))
print("Test =", len(test_ds))

Train = 400
Val = 50
Test = 50


In [13]:
model = ALIGNForRetrieval(embed_dim=512).to(device)
optimizer = AdamW(model.parameters(), lr=1e-5, weight_decay=1e-3)

best_val_loss = float("inf")
save_path = "/content/align_best.pth"

# =============================================================
# 7. TRAINING LOOP
# =============================================================
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        images = batch["images"].to(device)
        texts  = batch["texts"]

        logits, _, _ = model(images, texts)

        labels = torch.arange(logits.size(0)).to(device)
        loss_i2t = F.cross_entropy(logits, labels)
        loss_t2i = F.cross_entropy(logits.T, labels)
        loss = (loss_i2t + loss_t2i) / 2

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train = total_loss / len(train_loader)
    print(f"[Epoch {epoch+1}] Train Loss = {avg_train:.4f}")

    # ---------------- VAL ----------------
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for batch in val_loader:
            images = batch["images"].to(device)
            texts  = batch["texts"]

            logits, _, _ = model(images, texts)

            labels = torch.arange(logits.size(0)).to(device)
            loss = (
                F.cross_entropy(logits, labels) +
                F.cross_entropy(logits.T, labels)
            ) / 2

            val_loss += loss.item()

    val_loss /= len(val_loader)
    print("Val =", val_loss)

    # save best
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), save_path)
        print("🔥 Saved best checkpoint:", save_path)

Epoch 1: 100%|██████████| 25/25 [00:16<00:00,  1.56it/s]


[Epoch 1] Train Loss = 2.7127
Val = 2.2510443925857544
🔥 Saved best checkpoint: /content/align_best.pth


Epoch 2: 100%|██████████| 25/25 [00:15<00:00,  1.62it/s]


[Epoch 2] Train Loss = 2.4678
Val = 2.2716166377067566


Epoch 3: 100%|██████████| 25/25 [00:15<00:00,  1.62it/s]


...

[Epoch 8] Train Loss = 2.1385
Val = 2.09662726521492
🔥 Saved best checkpoint: /content/align_best.pth


Epoch 9: 100%|██████████| 25/25 [00:16<00:00,  1.52it/s]


[Epoch 9] Train Loss = 2.1317
Val = 2.085923060774803
🔥 Saved best checkpoint: /content/align_best.pth


Epoch 10: 100%|██████████| 25/25 [00:16<00:00,  1.48it/s]


[Epoch 10] Train Loss = 2.1220
Val = 2.0949717015028


In [14]:
model.load_state_dict(torch.load(save_path, map_location=device))
model.eval()

img_embeds = []
txt_embeds = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Extracting"):
        images = batch["images"].to(device)
        texts  = batch["texts"]

        _, img, txt = model(images, texts)
        img_embeds.append(img.cpu())
        txt_embeds.append(txt.cpu())

img_embeds = torch.cat(img_embeds)
txt_embeds = torch.cat(txt_embeds)

Extracting: 100%|██████████| 50/50 [00:01<00:00, 36.03it/s]


In [15]:
import numpy as np

# =========================================================
# Recall@K
# =========================================================
def recall_at_k(sim_matrix, k):
    N = sim_matrix.shape[0]
    k = min(k, N)

    ranking = np.argsort(-sim_matrix, axis=1)
    hits = np.array([1 if i in ranking[i, :k] else 0 for i in range(N)])

    return hits.mean()


# =========================================================
# Precision@K
# =========================================================
def precision_at_k(sim_matrix, k):
    N = sim_matrix.shape[0]
    k = min(k, N)

    ranking = np.argsort(-sim_matrix, axis=1)
    precisions = np.array([
        (1 if i in ranking[i, :k] else 0) / k
        for i in range(N)
    ])

    return precisions.mean()


# =========================================================
# Average Precision cho 1 sample
# =========================================================
def average_precision(ranking, gt_index):
    score = 0.0
    correct = 0
    for rank, idx in enumerate(ranking, start=1):
        if idx == gt_index:
            correct += 1
            score += correct / rank
    return score


# =========================================================
# Mean Average Precision (mAP)
# =========================================================
def mean_average_precision(sim_matrix):
    N = sim_matrix.shape[0]
    ranking = np.argsort(-sim_matrix, axis=1)

    aps = np.array([
        average_precision(ranking[i], i)
        for i in range(N)
    ])
    return aps.mean()


# =========================================================
# nDCG@K
# =========================================================
def ndcg_at_k(sim_matrix, k):
    N = sim_matrix.shape[0]
    k = min(k, N)

    ranking = np.argsort(-sim_matrix, axis=1)
    ndcgs = []

    for i in range(N):
        topk = ranking[i, :k]
        rel = np.array([1 if idx == i else 0 for idx in topk])

        dcg = np.sum(rel / np.log2(np.arange(2, k+2)))
        idcg = 1.0  # vì chỉ có 1 ground truth
        ndcgs.append(dcg / idcg)

    return np.mean(ndcgs)

In [16]:
sim_matrix = img_embeds @ txt_embeds.T

print("Recall@1 :", recall_at_k(sim_matrix, 1))
print("Recall@5 :", recall_at_k(sim_matrix, 5))
print("Recall@10:", recall_at_k(sim_matrix, 10))

print("Precision@1:", precision_at_k(sim_matrix, 1))
print("Precision@5:", precision_at_k(sim_matrix, 5))
print("Precision@10:", precision_at_k(sim_matrix, 10))


print("mAP:", mean_average_precision(sim_matrix))

print("nDCG@1:", ndcg_at_k(sim_matrix, 1))
print("nDCG@5:", ndcg_at_k(sim_matrix, 5))
print("nDCG@10:", ndcg_at_k(sim_matrix, 10))

Recall@1 : 0.08
Recall@5 : 0.4
Recall@10: 0.74
Precision@1: 0.08
Precision@5: 0.08
Precision@10: 0.07400000000000001
mAP: 0.24177620332883493
nDCG@1: 0.08
nDCG@5: 0.23479894588308953
nDCG@10: 0.34206940451368817


In [17]:
import numpy as np

def compute_mrr(sim_matrix):
    N = sim_matrix.shape[0]
    rr = []
    for i in range(N):
        ranks = np.argsort(-sim_matrix[i])
        pos = np.where(ranks == i)[0][0]
        rr.append(1 / (pos + 1))
    return np.mean(rr)

print("MRR (text → image):", compute_mrr(sim_matrix))

MRR (text → image): 0.24177620332883493
